In [1]:
import pandas as pd
import sqlite3

In [2]:
conn = sqlite3.connect('inventory.db')

In [3]:
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'",conn)
tables

,name
0,begin_inventory
1,end_inventory
2,purchases
3,purchase_prices
4,sales
5,vendor_invoice
6,vendor_sales_summary


In [5]:
output_folder = r"C:\Users\Rohan\PowerBI_Data"

for table in tables['name']:
    df = pd.read_sql(f"SELECT * FROM {table}", conn)
    df.to_csv(f"{output_folder}\\{table}.csv", index=False)

In [24]:
for table in tables['name']:
    print("count of records : " , pd.read_sql(f'SELECT COUNT(*) as count FROM {table}',conn)['count'].values[0])
    print(pd.read_sql(f'select * from {table} limit 5' , conn))
    

count of records :  206529
         InventoryId  Store          City  Brand                  Description  \
0  1_HARDERSFIELD_58      1  HARDERSFIELD     58  Gekkeikan Black & Gold Sake   
1  1_HARDERSFIELD_60      1  HARDERSFIELD     60       Canadian Club 1858 VAP   
2  1_HARDERSFIELD_62      1  HARDERSFIELD     62     Herradura Silver Tequila   
3  1_HARDERSFIELD_63      1  HARDERSFIELD     63   Herradura Reposado Tequila   
4  1_HARDERSFIELD_72      1  HARDERSFIELD     72         No. 3 London Dry Gin   

    Size  onHand  Price   startDate  
0  750mL       8  12.99  2024-01-01  
1  750mL       7  10.99  2024-01-01  
2  750mL       6  36.99  2024-01-01  
3  750mL       3  38.99  2024-01-01  
4  750mL       6  34.99  2024-01-01  
count of records :  224489
         InventoryId  Store          City  Brand                  Description  \
0  1_HARDERSFIELD_58      1  HARDERSFIELD     58  Gekkeikan Black & Gold Sake   
1  1_HARDERSFIELD_62      1  HARDERSFIELD     62     Herradura Silver

In [26]:
pd.read_sql('select * from purchases limit 5',conn)

,InventoryId,Store,Brand,Description,Size,VendorNumber,VendorName,PONumber,PODate,ReceivingDate,InvoiceDate,PayDate,PurchasePrice,Quantity,Dollars,Classification
0,69_MOUNTMEND_8412,69,8412,Tequila Ocho Plata Fresno,750mL,105,ALTAMAR BRANDS LLC,8124,2023-12-21,2024-01-02,2024-01-04,2024-02-16,35.71,6,214.26,1
1,30_CULCHETH_5255,30,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-01,2024-01-07,2024-02-21,9.35,4,37.40,1
2,34_PITMERDEN_5215,34,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-02,2024-01-07,2024-02-21,9.41,5,47.05,1
3,1_HARDERSFIELD_5255,1,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-01,2024-01-07,2024-02-21,9.35,6,56.10,1
4,76_DONCASTER_2034,76,2034,Glendalough Double Barrel,750mL,388,ATLANTIC IMPORTING COMPANY,8169,2023-12-24,2024-01-02,2024-01-09,2024-02-16,21.32,5,106.60,1


In [37]:
for table in tables['name']:
    if table == 'purchases':
        df = pd.read_sql(f"SELECT * FROM {table}", conn)

        print(
            df.groupby(['VendorNumber', 'VendorName'])[
                ['Quantity', 'Dollars', 'PurchasePrice']
            ].sum()
        )

                                                      Quantity    Dollars  \
VendorNumber VendorName                                                     
2            IRA GOLDMAN AND WILLIAMS, LLP                 328    5630.88   
54           AAPER ALCOHOL & CHEMICAL CO                     1     105.07   
60           ADAMBA IMPORTS INTL INC                      4732   76770.25   
105          ALTAMAR BRANDS LLC                            332   11706.20   
200          AMERICAN SPIRITS EXCHANGE                     132    1205.16   
...                                                        ...        ...   
98450        Serralles Usa LLC                           10463  168993.61   
99166        STARK BREWING COMPANY                        1212   25961.04   
172662       SWEETWATER FARM                              1629   34708.03   
173357       TAMWORTH DISTILLING                          1990   41036.44   
201359       FLAVOR ESSENCE INC                              1      17.00   

In [11]:
for table in tables['name']:
    if table == 'vendor_invoice':
        df = pd.read_sql(f"SELECT * FROM {table}", conn)
        display(df.head())

,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Approval
0,105,ALTAMAR BRANDS LLC,2024-01-04,8124,2023-12-21,2024-02-16,6,214.26,3.47,NaN
1,4466,AMERICAN VINTAGE BEVERAGE,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55,8.57,NaN
2,388,ATLANTIC IMPORTING COMPANY,2024-01-09,8169,2023-12-24,2024-02-16,5,106.60,4.61,NaN
3,480,BACARDI USA INC,2024-01-12,8106,2023-12-20,2024-02-05,10100,137483.78,2935.20,NaN
4,516,BANFI PRODUCTS CORP,2024-01-07,8170,2023-12-24,2024-02-12,1935,15527.25,429.20,NaN


In [13]:
frieght_summary = pd.read_sql("select VendorNumber , SUM(Freight) as frieght_sum from VENDOR_INVOICE GROUP BY VendorNumber",conn)
display(frieght_summary)

,VendorNumber,frieght_sum
0,2,27.08
1,54,0.48
2,60,367.52
3,105,62.39
4,200,6.19
...,...,...
121,98450,856.02
122,99166,130.09
123,172662,178.34
124,173357,202.50


In [14]:
for table in tables['name']:
    if table == 'purchase_prices' or table == 'purchases':
        df = pd.read_sql(f"SELECT * FROM {table}", conn)
        display(df.head())

,InventoryId,Store,Brand,Description,Size,VendorNumber,VendorName,PONumber,PODate,ReceivingDate,InvoiceDate,PayDate,PurchasePrice,Quantity,Dollars,Classification
0,69_MOUNTMEND_8412,69,8412,Tequila Ocho Plata Fresno,750mL,105,ALTAMAR BRANDS LLC,8124,2023-12-21,2024-01-02,2024-01-04,2024-02-16,35.71,6,214.26,1
1,30_CULCHETH_5255,30,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-01,2024-01-07,2024-02-21,9.35,4,37.40,1
2,34_PITMERDEN_5215,34,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-02,2024-01-07,2024-02-21,9.41,5,47.05,1
3,1_HARDERSFIELD_5255,1,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-01,2024-01-07,2024-02-21,9.35,6,56.10,1
4,76_DONCASTER_2034,76,2034,Glendalough Double Barrel,750mL,388,ATLANTIC IMPORTING COMPANY,8169,2023-12-24,2024-01-02,2024-01-09,2024-02-16,21.32,5,106.60,1


,Brand,Description,Price,Size,Volume,Classification,PurchasePrice,VendorNumber,VendorName
0,58,Gekkeikan Black & Gold Sake,12.99,750mL,750,1,9.28,8320,SHAW ROSS INT L IMP LTD
1,62,Herradura Silver Tequila,36.99,750mL,750,1,28.67,1128,BROWN-FORMAN CORP
2,63,Herradura Reposado Tequila,38.99,750mL,750,1,30.46,1128,BROWN-FORMAN CORP
3,72,No. 3 London Dry Gin,34.99,750mL,750,1,26.11,9165,ULTRA BEVERAGE COMPANY LLP
4,75,Three Olives Tomato Vodka,14.99,750mL,750,1,10.94,7245,PROXIMO SPIRITS INC.


In [24]:
display(pd.read_sql_query(f"""select
p.VendorNumber,
p.VendorName,
p.Brand,
p.PurchasePrice,
pp.Volume,
pp.Price as ActualPrice,
SUM(Quantity) as TotalPurchaseQuantity,
SUM(Dollars) as TotalPurchasedDollars
FROM purchases p
JOIN purchase_prices pp
ON p.Brand = pp.Brand
WHERE p.PurchasePrice > 0
GROUP BY p.VendorNumber,p.VendorName,p.Brand
ORDER BY TotalPurchasedDollars
""",conn))

,VendorNumber,VendorName,Brand,PurchasePrice,Volume,ActualPrice,TotalPurchaseQuantity,TotalPurchasedDollars
0,7245,PROXIMO SPIRITS INC.,3065,0.71,50,0.99,1,0.71
1,3960,DIAGEO NORTH AMERICA INC,6127,1.47,200,1.99,1,1.47
2,3924,HEAVEN HILL DISTILLERIES,9123,0.74,50,0.99,2,1.48
3,8004,SAZERAC CO INC,5683,0.39,50,0.49,6,2.34
4,9815,WINE GROUP INC,8527,1.32,750,4.99,2,2.64
...,...,...,...,...,...,...,...,...
10687,3960,DIAGEO NORTH AMERICA INC,3545,21.89,1750,29.99,138109,3023206.01
10688,3960,DIAGEO NORTH AMERICA INC,4261,16.17,1750,22.99,201682,3261197.94
10689,17035,PERNOD RICARD USA,8068,18.24,1750,24.99,187407,3418303.68
10690,4425,MARTIGNETTI COMPANIES,3405,23.19,1750,28.99,164038,3804041.22


In [ ]:
display(pd.read_sql("""select
VendorNo,
Brand,
SUM(SalesDollars) as TotalSalesDollars,
SUM(SalesPrice) as TotalSalesPrice,
SUM(SalesQuantity) as TotalSalesQuantity,
SUM(ExciseTax) as TotalExciseTax,
FROM sales
GROUP BY VendorNo , Brand
ORDER BY TotalSalesDollars
""",conn))

In [7]:
#joining three tables to get one table to implement our function to fetch useful Data
#optimized joining of Data

vendor_data_summary = pd.read_sql("""WITH FreightSummary AS(
Select 
VendorNumber ,
SUM(Freight) AS FreightCost
FROM vendor_invoice
GROUP BY VendorNumber
),

PurchaseSummary AS(
Select
p.VendorNumber,
p.VendorName,
p.Brand,
p.PurchasePrice,
p.Description,
pp.Volume,
pp.Price as ActualPrice,
SUM(Quantity) as TotalPurchasedQuantity,
SUM(Dollars) as TotalPurchasedDollars
FROM purchases p
JOIN purchase_prices pp
ON p.Brand = pp.Brand
WHERE p.PurchasePrice > 0
GROUP BY p.VendorNumber,p.VendorName,p.Brand
ORDER BY TotalPurchasedDollars
),

SalesSummary AS (
Select
VendorNo,
Brand,
SUM(SalesDollars) as TotalSalesDollars,
SUM(SalesPrice) as TotalSalesPrice,
SUM(SalesQuantity) as TotalSalesQuantity,
SUM(ExciseTax) as TotalExciseTax
FROM sales
GROUP BY VendorNo , Brand
ORDER BY TotalSalesDollars
)

Select
ps.VendorNumber,
ps.VendorName,
ps.Brand,
ps.Description,
ps.PurchasePrice,
ps.ActualPrice,
ps.Volume,
ps.TotalPurchasedQuantity,
ps.TotalPurchasedDollars,
ss.TotalSalesQuantity,
ss.TotalSalesDollars,
ss.TotalSalesPrice,
ss.TotalExciseTax,
fs.FreightCost
FROM PurchaseSummary ps
LEFT JOIN SalesSummary ss
    ON ps.VendorNumber = ss.VendorNo
    AND ps.Brand = ss.Brand
LEFT JOIN FreightSummary fs
    ON ps.VendorNumber = fs.VendorNumber
ORDER BY ps.TotalPurchasedDollars DESC""",conn)

In [8]:
vendor_data_summary

,VendorNumber,VendorName,Brand,Description,PurchasePrice,ActualPrice,Volume,TotalPurchasedQuantity,TotalPurchasedDollars,TotalSalesQuantity,TotalSalesDollars,TotalSalesPrice,TotalExciseTax,FreightCost
0,1128,BROWN-FORMAN CORP,1233,Jack Daniels No 7 Black,26.27,36.99,1750,145080,3811251.60,142049.0,5.101920e+06,672819.31,260999.20,68601.68
1,4425,MARTIGNETTI COMPANIES,3405,Tito's Handmade Vodka,23.19,28.99,1750,164038,3804041.22,160247.0,4.819073e+06,561512.37,294438.66,144929.24
2,17035,PERNOD RICARD USA,8068,Absolut 80 Proof,18.24,24.99,1750,187407,3418303.68,187140.0,4.538121e+06,461140.15,343854.07,123780.22
3,3960,DIAGEO NORTH AMERICA INC,4261,Capt Morgan Spiced Rum,16.17,22.99,1750,201682,3261197.94,200412.0,4.475973e+06,420050.01,368242.80,257032.07
4,3960,DIAGEO NORTH AMERICA INC,3545,Ketel One Vodka,21.89,29.99,1750,138109,3023206.01,135838.0,4.223108e+06,545778.28,249587.83,257032.07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10687,9815,WINE GROUP INC,8527,Concannon Glen Ellen Wh Zin,1.32,4.99,750,2,2.64,5.0,1.595000e+01,10.96,0.55,27100.41
10688,8004,SAZERAC CO INC,5683,Dr McGillicuddy's Apple Pie,0.39,0.49,50,6,2.34,134.0,6.566000e+01,1.47,7.04,50293.62
10689,3924,HEAVEN HILL DISTILLERIES,9123,Deep Eddy Vodka,0.74,0.99,50,2,1.48,2.0,1.980000e+00,0.99,0.10,14069.87
10690,3960,DIAGEO NORTH AMERICA INC,6127,The Club Strawbry Margarita,1.47,1.99,200,1,1.47,72.0,1.432800e+02,77.61,15.12,257032.07


In [9]:
vendor_data_summary.dtypes

VendorNumber                int64
VendorName                    str
Brand                       int64
Description                   str
PurchasePrice             float64
ActualPrice               float64
Volume                        str
TotalPurchasedQuantity      int64
TotalPurchasedDollars     float64
TotalSalesQuantity        float64
TotalSalesDollars         float64
TotalSalesPrice           float64
TotalExciseTax            float64
FreightCost               float64
dtype: object

In [13]:
vendor_data_summary.isnull().sum()

VendorNumber                0
VendorName                  0
Brand                       0
Description                 0
PurchasePrice               0
ActualPrice                 0
Volume                      0
TotalPurchasedQuantity      0
TotalPurchasedDollars       0
TotalSalesQuantity        178
TotalSalesDollars         178
TotalSalesPrice           178
TotalExciseTax            178
FreightCost                 0
dtype: int64

In [22]:
vendor_data_summary['VendorName']

0               BROWN-FORMAN CORP
1           MARTIGNETTI COMPANIES
2               PERNOD RICARD USA
3        DIAGEO NORTH AMERICA INC
4        DIAGEO NORTH AMERICA INC
                   ...           
10687              WINE GROUP INC
10688              SAZERAC CO INC
10689    HEAVEN HILL DISTILLERIES
10690    DIAGEO NORTH AMERICA INC
10691        PROXIMO SPIRITS INC.
Name: VendorName, Length: 10692, dtype: str

In [15]:
#resolving the inconsistencies in data

In [21]:
vendor_data_summary['VendorName'] = vendor_data_summary['VendorName'].str.strip()

In [18]:
vendor_data_summary['Volume'] = vendor_data_summary['Volume'].astype('float64')

In [19]:
vendor_data_summary.fillna(0 , inplace = True)

,VendorNumber,VendorName,Brand,Description,PurchasePrice,ActualPrice,Volume,TotalPurchasedQuantity,TotalPurchasedDollars,TotalSalesQuantity,TotalSalesDollars,TotalSalesPrice,TotalExciseTax,FreightCost
0,1128,BROWN-FORMAN CORP,1233,Jack Daniels No 7 Black,26.27,36.99,1750.0,145080,3811251.60,142049.0,5.101920e+06,672819.31,260999.20,68601.68
1,4425,MARTIGNETTI COMPANIES,3405,Tito's Handmade Vodka,23.19,28.99,1750.0,164038,3804041.22,160247.0,4.819073e+06,561512.37,294438.66,144929.24
2,17035,PERNOD RICARD USA,8068,Absolut 80 Proof,18.24,24.99,1750.0,187407,3418303.68,187140.0,4.538121e+06,461140.15,343854.07,123780.22
3,3960,DIAGEO NORTH AMERICA INC,4261,Capt Morgan Spiced Rum,16.17,22.99,1750.0,201682,3261197.94,200412.0,4.475973e+06,420050.01,368242.80,257032.07
4,3960,DIAGEO NORTH AMERICA INC,3545,Ketel One Vodka,21.89,29.99,1750.0,138109,3023206.01,135838.0,4.223108e+06,545778.28,249587.83,257032.07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10687,9815,WINE GROUP INC,8527,Concannon Glen Ellen Wh Zin,1.32,4.99,750.0,2,2.64,5.0,1.595000e+01,10.96,0.55,27100.41
10688,8004,SAZERAC CO INC,5683,Dr McGillicuddy's Apple Pie,0.39,0.49,50.0,6,2.34,134.0,6.566000e+01,1.47,7.04,50293.62
10689,3924,HEAVEN HILL DISTILLERIES,9123,Deep Eddy Vodka,0.74,0.99,50.0,2,1.48,2.0,1.980000e+00,0.99,0.10,14069.87
10690,3960,DIAGEO NORTH AMERICA INC,6127,The Club Strawbry Margarita,1.47,1.99,200.0,1,1.47,72.0,1.432800e+02,77.61,15.12,257032.07


In [20]:
vendor_data_summary.isnull().sum()

VendorNumber              0
VendorName                0
Brand                     0
Description               0
PurchasePrice             0
ActualPrice               0
Volume                    0
TotalPurchasedQuantity    0
TotalPurchasedDollars     0
TotalSalesQuantity        0
TotalSalesDollars         0
TotalSalesPrice           0
TotalExciseTax            0
FreightCost               0
dtype: int64

In [23]:
vendor_data_summary['VendorName'].unique()

<StringArray>
[          'BROWN-FORMAN CORP',       'MARTIGNETTI COMPANIES',
           'PERNOD RICARD USA',    'DIAGEO NORTH AMERICA INC',
             'BACARDI USA INC',     'JIM BEAM BRANDS COMPANY',
         'MAJESTIC FINE WINES',  'ULTRA BEVERAGE COMPANY LLP',
       'STOLI GROUP,(USA) LLC',        'PROXIMO SPIRITS INC.',
 ...
                    'UNCORKED',         'BRONCO WINE COMPANY',
     'MILTONS DISTRIBUTING CO',                'TRUETT HURST',
         'LAUREATE IMPORTS CO',     'FANTASY FINE WINES CORP',
 'AAPER ALCOHOL & CHEMICAL CO',      'SILVER MOUNTAIN CIDERS',
      'CAPSTONE INTERNATIONAL',          'FLAVOR ESSENCE INC']
Length: 128, dtype: str

In [24]:
#Adding columns required for EDA

In [25]:
vendor_data_summary['GrossProfit'] = vendor_data_summary['TotalSalesDollars'] - vendor_data_summary['TotalPurchasedDollars']

In [26]:
vendor_data_summary['ProfitMargin'] = (vendor_data_summary['GrossProfit'] / vendor_data_summary['TotalSalesDollars']) * 100

In [27]:
vendor_data_summary['StockTurnover'] = vendor_data_summary['TotalSalesQuantity'] / vendor_data_summary['TotalPurchasedQuantity']

In [28]:
vendor_data_summary['SalestoPurchaseRatio'] = vendor_data_summary['TotalSalesDollars'] / vendor_data_summary['TotalPurchasedDollars']

In [29]:
#Create an empty Table then insert Data into it

In [30]:
cursor = conn.cursor()

In [31]:
vendor_data_summary.columns

Index(['VendorNumber', 'VendorName', 'Brand', 'Description', 'PurchasePrice',
       'ActualPrice', 'Volume', 'TotalPurchasedQuantity',
       'TotalPurchasedDollars', 'TotalSalesQuantity', 'TotalSalesDollars',
       'TotalSalesPrice', 'TotalExciseTax', 'FreightCost', 'GrossProfit',
       'ProfitMargin', 'StockTurnover', 'SalestoPurchaseRatio'],
      dtype='str')

In [32]:
cursor.execute("""CREATE TABLE vendor_sales_summary(
VendorNumber INT,
VendorName VARCHAR(100),
Brand INT,
Description VARCHAR(100),
PurchasePrice DECIMAL(10,2),
ActualPrice DECIMAL(10,2),
Volume FLOAT,
TotalPurchasedQuantity INT,
TotalPurchasedDollars DECIMAL(15,2),
TotalSalesQuantity INT,
TotalSalesDollars DECIMAL(15,2),
TotalSalesPrice DECIMAL(15,2),
TotalExciseTax DECIMAL(15,2),
FreightCost DECIMAL(15,2),
GrossProfit DECIMAL(15,2),
ProfitMargin DECIMAL(15,2),
StockTurnover DECIMAL(15,2),
SalestoPurchaseRatio DECIMAL(15,2),
PRIMARY KEY(VendorNumber , Brand)
);""")

In [33]:
pd.read_sql("""select * from vendor_sales_summary""",conn)

,VendorNumber,VendorName,Brand,Description,PurchasePrice,ActualPrice,Volume,TotalPurchasedQuantity,TotalPurchasedDollars,TotalSalesQuantity,TotalSalesDollars,TotalSalesPrice,TotalExciseTax,FreightCost,GrossProfit,ProfitMargin,StockTurnover,SalestoPurchaseRatio


In [36]:
vendor_data_summary.to_sql('vendor_sales_summary' , conn , if_exists = 'replace' , index = False)

10692

In [37]:
pd.read_sql("""select * from vendor_sales_summary""",conn)

,VendorNumber,VendorName,Brand,Description,PurchasePrice,ActualPrice,Volume,TotalPurchasedQuantity,TotalPurchasedDollars,TotalSalesQuantity,TotalSalesDollars,TotalSalesPrice,TotalExciseTax,FreightCost,GrossProfit,ProfitMargin,StockTurnover,SalestoPurchaseRatio
0,1128,BROWN-FORMAN CORP,1233,Jack Daniels No 7 Black,26.27,36.99,1750.0,145080,3811251.60,142049.0,5.101920e+06,672819.31,260999.20,68601.68,1290667.91,25.297693,0.979108,1.338647
1,4425,MARTIGNETTI COMPANIES,3405,Tito's Handmade Vodka,23.19,28.99,1750.0,164038,3804041.22,160247.0,4.819073e+06,561512.37,294438.66,144929.24,1015032.27,21.062810,0.976890,1.266830
2,17035,PERNOD RICARD USA,8068,Absolut 80 Proof,18.24,24.99,1750.0,187407,3418303.68,187140.0,4.538121e+06,461140.15,343854.07,123780.22,1119816.92,24.675786,0.998575,1.327594
3,3960,DIAGEO NORTH AMERICA INC,4261,Capt Morgan Spiced Rum,16.17,22.99,1750.0,201682,3261197.94,200412.0,4.475973e+06,420050.01,368242.80,257032.07,1214774.94,27.139908,0.993703,1.372493
4,3960,DIAGEO NORTH AMERICA INC,3545,Ketel One Vodka,21.89,29.99,1750.0,138109,3023206.01,135838.0,4.223108e+06,545778.28,249587.83,257032.07,1199901.61,28.412764,0.983556,1.396897
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10687,9815,WINE GROUP INC,8527,Concannon Glen Ellen Wh Zin,1.32,4.99,750.0,2,2.64,5.0,1.595000e+01,10.96,0.55,27100.41,13.31,83.448276,2.500000,6.041667
10688,8004,SAZERAC CO INC,5683,Dr McGillicuddy's Apple Pie,0.39,0.49,50.0,6,2.34,134.0,6.566000e+01,1.47,7.04,50293.62,63.32,96.436186,22.333333,28.059829
10689,3924,HEAVEN HILL DISTILLERIES,9123,Deep Eddy Vodka,0.74,0.99,50.0,2,1.48,2.0,1.980000e+00,0.99,0.10,14069.87,0.50,25.252525,1.000000,1.337838
10690,3960,DIAGEO NORTH AMERICA INC,6127,The Club Strawbry Margarita,1.47,1.99,200.0,1,1.47,72.0,1.432800e+02,77.61,15.12,257032.07,141.81,98.974037,72.000000,97.469388


In [39]:
import pandas as pd
import sqlite3
import logging
from ingestion_db import ingest_db

logging.basicConfig(
    filename = "logs/get_vendor_summary.log",
    level = logging.DEBUG,
    format = "%(asctime)s - %(levelname)s - %(message)s",
    filemode = "a"
)

def create_vendor_summary(conn):
    '''this function would join the tables and give one resultant table and add new columns into table'''
    vendor_sales_summary = pd.read_sql("""WITH FreightSummary AS(
    Select 
    VendorNumber ,
    SUM(Freight) AS FreightCost
    FROM vendor_invoice
    GROUP BY VendorNumber
    ),
    
    PurchaseSummary AS(
    Select
    p.VendorNumber,
    p.VendorName,
    p.Brand,
    p.PurchasePrice,
    p.Description,
    pp.Volume,
    pp.Price as ActualPrice,
    SUM(Quantity) as TotalPurchasedQuantity,
    SUM(Dollars) as TotalPurchasedDollars
    FROM purchases p
    JOIN purchase_prices pp
    ON p.Brand = pp.Brand
    WHERE p.PurchasePrice > 0
    GROUP BY p.VendorNumber,p.VendorName,p.Brand
    ORDER BY TotalPurchasedDollars
    ),
    
    SalesSummary AS (
    Select
    VendorNo,
    Brand,
    SUM(SalesDollars) as TotalSalesDollars,
    SUM(SalesPrice) as TotalSalesPrice,
    SUM(SalesQuantity) as TotalSalesQuantity,
    SUM(ExciseTax) as TotalExciseTax
    FROM sales
    GROUP BY VendorNo , Brand
    ORDER BY TotalSalesDollars
    )
    
    Select
    ps.VendorNumber,
    ps.VendorName,
    ps.Brand,
    ps.Description,
    ps.PurchasePrice,
    ps.ActualPrice,
    ps.Volume,
    ps.TotalPurchasedQuantity,
    ps.TotalPurchasedDollars,
    ss.TotalSalesQuantity,
    ss.TotalSalesDollars,
    ss.TotalSalesPrice,
    ss.TotalExciseTax,
    fs.FreightCost
    FROM PurchaseSummary ps
    LEFT JOIN SalesSummary ss
        ON ps.VendorNumber = ss.VendorNo
        AND ps.Brand = ss.Brand
    LEFT JOIN FreightSummary fs
        ON ps.VendorNumber = fs.VendorNumber
    ORDER BY ps.TotalPurchasedDollars DESC""",conn)

    return vendor_sales_summary
    
def clean_data(df):
    '''this function will clean the Data'''
    #removing spaces from categorical columns
    df['VendorName'] = df['VendorName'].str.strip()
    df['Description'] = df['Description'].str.strip()

    #filling missing value with 0
    df.fillna(0 , inplace = True)

    #Changing datatype to float
    df['Volume'] = df['Volume'].astype('float')

    #Creating new columns for better Analysis
    df['GrossProfit'] = df['TotalSalesDollars'] - df['TotalPurchasedDollars']
    df['ProfitMargin'] = (df['GrossProfit'] / df['TotalSalesDollars']) * 100
    df['StockTurnover'] = df['TotalSalesQuantity'] / df['TotalPurchasedQuantity']
    df['SalestoPurchaseRatio'] = df['TotalSalesDollars'] / df['TotalPurchasedDollars']

    return df
if __name__ == '__main__':
    #creating database connection
    conn = sqlite3.connect('inventory.db')

    logging.info('Creating Vendor Summary Table....')
    summary_df = create_vendor_summary(conn)
    logging.info(summary_df.head())

    logging.info('Cleaning Data....')
    clean_df = clean_data(summary_df)
    logging.info(clean_df.head())

    logging.info('Ingesting Data....')
    ingest_db(clean_df ,'vendor_sales_summary' ,conn)
    logging.info("completed!")

In [43]:
import pandas as pd
import sqlite3
import logging
import os
from ingestion_db import ingest_db

os.makedirs("logs", exist_ok=True)

logging.basicConfig(
    filename = "logs/get_vendor_summary.log",
    level = logging.DEBUG,
    format = "%(asctime)s - %(levelname)s - %(message)s",
    filemode = "a"
)

def create_vendor_summary(conn):
    '''this function would join the tables and give one resultant table and add new columns into table'''
    vendor_sales_summary = pd.read_sql("""WITH FreightSummary AS(
    Select 
    VendorNumber ,
    SUM(Freight) AS FreightCost
    FROM vendor_invoice
    GROUP BY VendorNumber
    ),
    
    PurchaseSummary AS(
    Select
    p.VendorNumber,
    p.VendorName,
    p.Brand,
    p.PurchasePrice,
    p.Description,
    pp.Volume,
    pp.Price as ActualPrice,
    SUM(Quantity) as TotalPurchasedQuantity,
    SUM(Dollars) as TotalPurchasedDollars
    FROM purchases p
    JOIN purchase_prices pp
    ON p.Brand = pp.Brand
    WHERE p.PurchasePrice > 0
    GROUP BY p.VendorNumber,p.VendorName,p.Brand
    ORDER BY TotalPurchasedDollars
    ),
    
    SalesSummary AS (
    Select
    VendorNo,
    Brand,
    SUM(SalesDollars) as TotalSalesDollars,
    SUM(SalesPrice) as TotalSalesPrice,
    SUM(SalesQuantity) as TotalSalesQuantity,
    SUM(ExciseTax) as TotalExciseTax
    FROM sales
    GROUP BY VendorNo , Brand
    ORDER BY TotalSalesDollars
    )
    
    Select
    ps.VendorNumber,
    ps.VendorName,
    ps.Brand,
    ps.Description,
    ps.PurchasePrice,
    ps.ActualPrice,
    ps.Volume,
    ps.TotalPurchasedQuantity,
    ps.TotalPurchasedDollars,
    ss.TotalSalesQuantity,
    ss.TotalSalesDollars,
    ss.TotalSalesPrice,
    ss.TotalExciseTax,
    fs.FreightCost
    FROM PurchaseSummary ps
    LEFT JOIN SalesSummary ss
        ON ps.VendorNumber = ss.VendorNo
        AND ps.Brand = ss.Brand
    LEFT JOIN FreightSummary fs
        ON ps.VendorNumber = fs.VendorNumber
    ORDER BY ps.TotalPurchasedDollars DESC""",conn)

    return vendor_sales_summary
    
def clean_data(df):
    '''this function will clean the Data'''
    #removing spaces from categorical columns
    df['VendorName'] = df['VendorName'].str.strip()
    df['Description'] = df['Description'].str.strip()

    #filling missing value with 0
    df.fillna(0 , inplace = True)

    #Changing datatype to float
    df['Volume'] = df['Volume'].astype('float')

    #Creating new columns for better Analysis
    df['GrossProfit'] = df['TotalSalesDollars'] - df['TotalPurchasedDollars']
    df['ProfitMargin'] = (df['GrossProfit'] / df['TotalSalesDollars']) * 100
    df['StockTurnover'] = df['TotalSalesQuantity'] / df['TotalPurchasedQuantity']
    df['SalestoPurchaseRatio'] = df['TotalSalesDollars'] / df['TotalPurchasedDollars']

    return df
if __name__ == '__main__':
    #creating database connection
    conn = sqlite3.connect('inventory.db')

    logging.info('Creating Vendor Summary Table....')
    summary_df = create_vendor_summary(conn)
    logging.info(summary_df.head())

    logging.info('Cleaning Data....')
    clean_df = clean_data(summary_df)
    logging.info(clean_df.head())

    logging.info('Ingesting Data....')
    ingest_db(clean_df ,'vendor_sales_summary' ,conn)
    logging.info("completed!")

    